# GIDS Observer Framework — Overview

This notebook is the front door.

The paper is ambitious. The code should not be mysterious.

The operational question is smaller and cleaner:
**can we represent a person-side slow embedding, a fast task-conditioned state, a proposition, and a transition map in a way another ML team can actually inspect, benchmark, and reuse?**

That is what this project is for.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
elif not (project_root / "src").exists() and (project_root.parent / "src").exists():
    project_root = project_root.parent

src_dir = project_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

results_dir = project_root / "results"
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", 50)

In [2]:
from gids_observer_framework.references import PAPER_REFERENCE_TABLE

ref_df = pd.DataFrame(PAPER_REFERENCE_TABLE())
ref_df

,key,part,section,equation,implementation,why_it_matters
0,salience_slice,Part 2,"Dimensionality Reduction, Attention, and Relevance","z_{i,t} = a_{i,t} \odot \pi_\tau(T_i, c_{i,t}, x_t)",embedding.salience_slice,"The transition often depends on a weighted slice of the person-space, not the whole thing at once."
1,predictive_state,Part 2 / Part 3 / Part 4,The Notion of State / Towards a Universal State Transition Function / Operational Definition of State,"P(Y\mid H, T, c, w, x) = P(Y\mid q, x)",experiments.run_toy_equations.predictive_sufficiency_demo,"This is the honest formal target: task-conditioned predictive state, not the whole ineffable interior."
2,memory_field,Part 2,Memory as a Series of Vectors,"m_{i,t} = \sum_j \omega_{ij,t} \mu_{ij}",memory.memory_field,Repeated traces become a weighted field instead of a vague story about memory.
3,contextual_lift,Part 2,Categorical Trace Pooling as an Operational Memory Estimator,"\widetilde{C}_{i,t}^{(f,s)} = \Xi(C_{i,t}^{(f,s)}, c_{i,t})",categorical.contextual_lift,Surface contradictions often disappear once you type the asymmetry carried by the regime.
4,categorical_pooling,Part 2,Categorical Trace Pooling as an Operational Memory Estimator,"u_{i,t}^{(f,s)} = \frac{1}{|\widetilde{C}|} \sum_{c \in \widetilde{C}} E_{f,s}(c)",categorical.build_event_categorical_embedding,Sparse categorical traces become fixed-width vectors without pretending missing slots do not exist.
5,slow_bank,Part 2,Categorical Trace Pooling as an Operational Memory Estimator,"g_{i,\rho}^{\mathrm{slow}} = \text{weighted regime average of } e_{i,r}^{\mathrm{cat}}",categorical.build_slow_bank,"Durable, regime-aware person structure should not be collapsed into one global mush."
6,fast_pool,Part 2,Categorical Trace Pooling as an Operational Memory Estimator,"g_{i,t}^{\mathrm{fast},\tau} = \sum_{r \le t} \alpha_{i,r,t}^{(\tau)} e_{i,r}^{\mathrm{cat}}",categorical.build_fast_pool,"Recent decisive traces and accumulated weak exposure both matter, but not equally."
7,slow_embedding,Part 2,Deriving the Transcendental Embedding,"\hat T_i^{(0)} = E(p_i, b_i, \ell_i, r_i, h_i, g_i^{\mathrm{slow}})",embedding.estimate_slow_embedding,This is the first operational estimate of the person-side embedding.
8,world_model,Part 3 / Part 4,Creating the World Model / The Proposed Latent-State Model,"\hat q_{i,t+1}^{(\tau,\Delta)} = F_\tau(\hat T_i, z_{i,t}, c_{i,t}, w_t, x_t)",state.world_model_step,A world model here is a simulator of predictive-state transitions under propositions.
9,readout,Part 3 / Part 4,Towards a Universal State Transition Function / The Proposed Latent-State Model,"\hat y_{i,t+\Delta}^{(\tau)} = R_0(\hat q_{i,t+1}^{(\tau,\Delta)})",state.readout_probability,"Observed outcomes are visible residues of the transition, not the state itself."


## Notebook map

1. **01_part2_embedding_and_memory.ipynb**  
   Turns the Part 2 machinery into code: salience, predictive state, memory as weighted traces, contextual lifting, categorical pooling, slow bank, and fast pool.

2. **02_part3_world_model_and_search.ipynb**  
   Covers the world model, the corrected training objective, projection equivalence, and proposition search.

3. **03_part4_benchmarking_and_results.ipynb**  
   Runs the temporal benchmark, the ablations, and the IPS demo.

4. **04_ml_team_adaptation_guide.ipynb**  
   Shows how another team should adapt the framework without cargo-culting the toy setup.

In [3]:
import os

summary = []
for folder in ["src/gids_observer_framework", "docs", "results", "notebooks"]:
    path = project_root / folder
    summary.append(
        {
            "folder": folder,
            "example_files": sorted(os.listdir(path))[:8],
        }
    )
pd.DataFrame(summary)

,folder,example_files
0,src/gids_observer_framework,"[__init__.py, __pycache__, benchmark.py, categorical.py, embedding.py, experiments, math_utils.py, memory.py]"
1,docs,"[paper_to_code_map.md, results_and_applicability.md]"
2,results,"[benchmark_cold_start_results.csv, benchmark_main_results.csv, benchmark_person_holdout_results.csv, benchmark_probe..."
3,notebooks,"[00_framework_overview.ipynb, 01_part2_embedding_and_memory.ipynb, 02_part3_world_model_and_search.ipynb, 03_part4_b..."


## What was patched from the draft

The project standardizes on two corrected equations.

**Training objective under gradient descent**
\[
\mathcal L_\tau
=
\mathcal L_{\mathrm{main}}
+
\sum_m \lambda_m \mathcal L_{\mathrm{probe},m}
+
\lambda_{\mathrm{reg}}\Omega(\theta)
\]

**Slow embedding refresh**
\[
\hat T_i \leftarrow (1-\alpha)\hat T_i + \alpha\hat T_i^{\mathrm{new}}
\]

The later notebooks show why those two are the keepers.